In [10]:
pip install scikit-learn

     ---------------------------------------- 8.1/8.1 MB 9.2 MB/s eta 0:00:00
     ---------------------------------------- 36.6/36.6 MB 9.8 MB/s eta 0:00:00
     ------------------------------------- 309.1/309.1 kB 19.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import numpy as np

# Load the data
D = np.genfromtxt("data/lines.csv", delimiter=",", skip_header=1)

# Extract data for the first line (x1 and y1)
x1 = D[:, 0]
y1 = D[:, 3]

def total_least_squares(x, y):
    # Center the data
    x_mean = np.mean(x)
    y_mean = np.mean(y)
    x_centered = x - x_mean
    y_centered = y - y_mean
    
    # Construct the matrix for SVD
    A = np.vstack([x_centered, y_centered]).T
    
    # Perform SVD
    _, _, Vh = np.linalg.svd(A)
    

    a, b = Vh[-1, :]
    
    # Solve for slope and intercept
    slope = -a / b
    intercept = y_mean - slope * x_mean
    
    return slope, intercept

m1, c1 = total_least_squares(x1, y1)
print(f"First Line Parameters: Slope = {m1:.4f}, Intercept = {c1:.4f}")

First Line Parameters: Slope = 1.2207, Intercept = -5.9872


In [ ]:
from sklearn.linear_model import RANSACRegressor

# Prepare the flattened data
X_cols = D[:, :3]
Y_cols = D[:, 3:]
X_all = X_cols.flatten().reshape(-1, 1)
Y_all = Y_cols.flatten()

# Storage for results
remaining_X = X_all
remaining_Y = Y_all
lines = []

for i in range(3):
    # Initialize RANSAC
    ransac = RANSACRegressor(residual_threshold=0.5)
    ransac.fit(remaining_X, remaining_Y)
    
    # Identify the inliers
    inlier_mask = ransac.inlier_mask_
    outlier_mask = np.logical_not(inlier_mask)
    
    # Store the parameters
    m = ransac.estimator_.coef_[0]
    b = ransac.estimator_.intercept_
    lines.append((m, b))
    
    print(f"Line {i+1}: Slope = {m:.4f}, Intercept = {b:.4f}")
    
    # Remove the points accounted for
    remaining_X = remaining_X[outlier_mask]
    remaining_Y = remaining_Y[outlier_mask]

    if len(remaining_X) == 0:
        break

Line 1: Slope = -0.4496, Intercept = 2.1993
Line 2: Slope = 1.0287, Intercept = 1.0070
Line 3: Slope = 1.2734, Intercept = -6.0581


In [4]:
import cv2
import numpy as np

points = []
def mouse_callback(event, x, y, flags, param):
    global points, img_display
    if event == cv2.EVENT_LBUTTONDOWN:
        if len(points) < 4:
            points.append((x, y))
        
            print(f"Point {len(points)}: ({x}, {y})")
            cv2.circle(img_display, (x, y), 5, (0, 0, 255), -1)
            cv2.imshow("Image", img_display)
            
            if len(points) == 4:
                print("\nFour points selected:")
                for i, p in enumerate(points):
                    print(f"P{i+1}: {p}")
                print("Press any key to process the flag.")

# Load the images
img = cv2.imread("data/turf.jpg")
flag = cv2.imread("data/flag.jpg")

if img is None or flag is None:
    raise FileNotFoundError("Make sure both 'turf.jpg' and 'flag.jpg' are in the folder.")

img_display = img.copy()
cv2.namedWindow("Image")
cv2.setMouseCallback("Image", mouse_callback)
cv2.imshow("Image", img_display)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Convert list to numpy array as per your snippet
points = np.array(points, dtype=np.float32)


Point 1: (687, 163)
Point 2: (841, 161)
Point 3: (1384, 683)
Point 4: (88, 667)

Four points selected:
P1: (687, 163)
P2: (841, 161)
P3: (1384, 683)
P4: (88, 667)
Press any key to process the flag.


In [ ]:

if len(points) == 4:
    # Define the dimensions of the flag
    h_f, w_f = flag.shape[:2]

    # Define the source coordinates 
    src_pts = np.array([
        [0, 0],
        [w_f - 1, 0],
        [w_f - 1, h_f - 1],
        [0, h_f - 1]
    ], dtype=np.float32)

    # Calculate the Perspective Transformation Matrix
    matrix = cv2.getPerspectiveTransform(src_pts, points)

    # Warp the flag to the perspective of the turf
    warped_flag = cv2.warpPerspective(flag, matrix, (img.shape[1], img.shape[0]))

    # Create a mask to cut a hole in the turf image
    mask = np.zeros_like(img, dtype=np.uint8)
    cv2.fillConvexPoly(mask, points.astype(int), (255, 255, 255))
    
    # Extract the background
    turf_background = cv2.bitwise_and(img, cv2.bitwise_not(mask))
    
    # Extract the foreground (the warped flag area only)
    flag_foreground = cv2.bitwise_and(warped_flag, mask)

    # Combine them
    final_result = cv2.add(turf_background, flag_foreground)

    # Display results
    cv2.imshow("Result", final_result)
    cv2.imwrite("data/superimposed_flag.jpg", final_result)
    print("\nSuccess! Result saved as 'superimposed_flag.jpg'")
    cv2.waitKey(0)
    cv2.destroyAllWindows()
else:
    print("Error: You didn't select exactly 4 points.")


Success! Result saved as 'superimposed_flag.jpg'
